# Аудит

In [5]:
# ============================================================
# АНАЛИЗ ВЫБРАННЫХ КЛАССОВ
# И ОПРЕДЕЛЕНИЕ ОБЪЁМА ДЛЯ ИТОГОВОГО DATASET
# ============================================================
#
# ЛОГИКА:
#
# 1. Для каждого датасета читается data.yaml / dataset.yaml.
#
# 2. Через CLASS_MAPPING определяются только выбранные классы.
#
# 3. Для каждого исходного класса считаются:
#       - количество объектов;
#       - количество уникальных изображений.
#
# 4. Формируется сводка по итоговым классам.
#
# 5. Три типа кранов:
#       Подъёмный кран
#       Стационарный кран
#       Автокран
#
#    рассматриваются отдельно.
#
# 6. Для кранов балансировка выполняется ПО ИЗОБРАЖЕНИЯМ:
#
#       target_images =
#       минимальное количество изображений
#       среди трёх типов кранов.
#
# 7. Например:
#
#       Подъёмный кран      3781
#       Стационарный кран   3135
#       Автокран            1791
#
#       target_images = 1791
#
#    Следовательно, при сборке итогового dataset
#    необходимо взять:
#
#       Подъёмный кран      1791 изображение
#       Стационарный кран   1791 изображение
#       Автокран            1791 изображение
#
# 8. Три класса затем будут объединены в один:
#
#       Кран
#
# ВАЖНО:
# этот скрипт пока НЕ создаёт итоговый YOLO dataset.
# Он только рассчитывает статистику и параметры отбора.
#
# ============================================================


from pathlib import Path
from collections import defaultdict
import pandas as pd
import yaml


# ============================================================
# НАСТРОЙКИ
# ============================================================

DATASETS_ROOT = Path(
    r"C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets"
)

OUTPUT_ROOT = Path(
    r"C:\Users\Ева\Desktop\Хакатон\dataset"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CLASS_MAPPING
# ============================================================
#
# ВАЖНО:
# вставь сюда свой полный CLASS_MAPPING.
#
# Формат:
#
# (название_датасета, исходный_класс): итоговый_класс
#
# ============================================================

CLASS_MAPPING = {

    # --------------------------------------------------------
    # -test- 2000-3000.v1i.yolo26
    # --------------------------------------------------------

    (
        "-test- 2000-3000.v1i.yolo26",
        "Bulldozer"
    ): "Бульдозер",

    (
        "-test- 2000-3000.v1i.yolo26",
        "Concrete mixer"
    ): "Автобетоносмеситель",

    (
        "-test- 2000-3000.v1i.yolo26",
        "Crane"
    ): "Подъёмный кран",

    (
        "-test- 2000-3000.v1i.yolo26",
        "Excavator"
    ): "Экскаватор",

    (
        "-test- 2000-3000.v1i.yolo26",
        "Pump truck"
    ): "Автобетононасос",

    (
        "-test- 2000-3000.v1i.yolo26",
        "Roller"
    ): "Каток",

    (
        "-test- 2000-3000.v1i.yolo26",
        "Static crane"
    ): "Стационарный кран",

    (
        "-test- 2000-3000.v1i.yolo26",
        "Truck"
    ): "Самосвал",


    # --------------------------------------------------------
    # 20230701_HeavyEquipment.v1i.yolo26
    # --------------------------------------------------------

    (
        "20230701_HeavyEquipment.v1i.yolo26",
        "Bulldozer"
    ): "Бульдозер",

    (
        "20230701_HeavyEquipment.v1i.yolo26",
        "ConcreteMixer"
    ): "Автобетоносмеситель",

    (
        "20230701_HeavyEquipment.v1i.yolo26",
        "Crane"
    ): "Подъёмный кран",

    (
        "20230701_HeavyEquipment.v1i.yolo26",
        "Excavator"
    ): "Экскаватор",

    (
        "20230701_HeavyEquipment.v1i.yolo26",
        "PumpTruck"
    ): "Автобетононасос",

    (
        "20230701_HeavyEquipment.v1i.yolo26",
        "Roller"
    ): "Каток",

    (
        "20230701_HeavyEquipment.v1i.yolo26",
        "StaticCrane"
    ): "Стационарный кран",

    (
        "20230701_HeavyEquipment.v1i.yolo26",
        "Truck"
    ): "Самосвал",


    # --------------------------------------------------------
    # constraction_yolo
    # --------------------------------------------------------

    (
        "constraction_yolo",
        "бетономешалка"
    ): "Автобетоносмеситель",

    (
        "constraction_yolo",
        "грузовик"
    ): "Самосвал",


    # --------------------------------------------------------
    # dataset_1
    # --------------------------------------------------------

    (
        "dataset_1",
        "Dump truck"
    ): "Самосвал",

    (
        "dataset_1",
        "Excavator"
    ): "Экскаватор",

    (
        "dataset_1",
        "Roller"
    ): "Каток",

    (
        "dataset_1",
        "Gazelle"
    ): "Грузовой автомобиль",

    (
        "dataset_1",
        "Bucket loader Big"
    ): "Фронтальный погрузчик",

    (
        "dataset_1",
        "Mixer"
    ): "Автобетоносмеситель",

    (
        "dataset_1",
        "Bulldozer"
    ): "Бульдозер",

    (
        "dataset_1",
        "Truck"
    ): "Грузовой автомобиль",

    (
        "dataset_1",
        "Trailer"
    ): "Прицеп",

    (
        "dataset_1",
        "Bucket loader Standart"
    ): "Фронтальный погрузчик",

    (
        "dataset_1",
        "Autocran"
    ): "Автокран",


    # --------------------------------------------------------
    # dataset_2
    # --------------------------------------------------------

    (
        "dataset_2",
        "crane"
    ): "Подъёмный кран",

    (
        "dataset_2",
        "excavator"
    ): "Экскаватор",

    (
        "dataset_2",
        "truck"
    ): "Самосвал",


    # --------------------------------------------------------
    # FYP_MOCSdataset.v3i.yolo26_bbox
    # --------------------------------------------------------

    (
        "FYP_MOCSdataset.v3i.yolo26_bbox",
        "Bulldozer"
    ): "Бульдозер",

    (
        "FYP_MOCSdataset.v3i.yolo26_bbox",
        "Concrete mixer"
    ): "Автобетоносмеситель",

    (
        "FYP_MOCSdataset.v3i.yolo26_bbox",
        "Crane"
    ): "Подъёмный кран",

    (
        "FYP_MOCSdataset.v3i.yolo26_bbox",
        "Excavator"
    ): "Экскаватор",

    (
        "FYP_MOCSdataset.v3i.yolo26_bbox",
        "Pump truck"
    ): "Автобетононасос",

    (
        "FYP_MOCSdataset.v3i.yolo26_bbox",
        "Roller"
    ): "Каток",

    (
        "FYP_MOCSdataset.v3i.yolo26_bbox",
        "Static crane"
    ): "Стационарный кран",

    (
        "FYP_MOCSdataset.v3i.yolo26_bbox",
        "Truck"
    ): "Самосвал",


    # --------------------------------------------------------
    # Heavy Equipment_yolo
    # --------------------------------------------------------

    (
        "Heavy Equipment_yolo",
        "bulldozer"
    ): "Бульдозер",

    (
        "Heavy Equipment_yolo",
        "dump"
    ): "Самосвал",

    (
        "Heavy Equipment_yolo",
        "excavator"
    ): "Экскаватор",

    (
        "Heavy Equipment_yolo",
        "roller"
    ): "Каток",


    # --------------------------------------------------------
    # large-machinery-v3.v1i.yolo26
    # --------------------------------------------------------

    (
        "large-machinery-v3.v1i.yolo26",
        "excavator"
    ): "Экскаватор",

    (
        "large-machinery-v3.v1i.yolo26",
        "crane"
    ): "Подъёмный кран",


    # --------------------------------------------------------
    # yolo_portable crane.v1i.yolo26
    # --------------------------------------------------------

    (
        "yolo_portable crane.v1i.yolo26",
        "Crane"
    ): "Подъёмный кран",
}


# ============================================================
# НАСТРОЙКИ КРАНОВ
# ============================================================

CRANE_CLASSES = [
    "Подъёмный кран",
    "Стационарный кран",
    "Автокран",
]

CRANE_FINAL_CLASS = "Кран"


# ============================================================
# ПОИСК YAML
# ============================================================

def find_dataset_yaml(dataset_dir):

    for filename in [
        "data.yaml",
        "dataset.yaml"
    ]:

        path = dataset_dir / filename

        if path.exists():
            return path

    return None


# ============================================================
# ЧТЕНИЕ КЛАССОВ ИЗ YAML
# ============================================================

def read_yaml_classes(dataset_dir):

    yaml_path = find_dataset_yaml(
        dataset_dir
    )

    if yaml_path is None:

        raise FileNotFoundError(
            f"\nНе найден data.yaml/dataset.yaml:\n"
            f"{dataset_dir}"
        )


    with open(
        yaml_path,
        "r",
        encoding="utf-8"
    ) as f:

        data = yaml.safe_load(f)


    names = data.get("names")


    if isinstance(names, dict):

        return {
            int(class_id): str(class_name)
            for class_id, class_name
            in names.items()
        }


    if isinstance(names, list):

        return {
            class_id: str(class_name)
            for class_id, class_name
            in enumerate(names)
        }


    raise ValueError(
        f"Не удалось определить names:\n"
        f"{yaml_path}"
    )


# ============================================================
# ПОИСК ИЗОБРАЖЕНИЙ
# ============================================================

def get_image_files(images_dir):

    image_files = []

    for extension in [
        "*.jpg",
        "*.jpeg",
        "*.png",
        "*.bmp",
        "*.webp",
    ]:

        image_files.extend(
            images_dir.rglob(extension)
        )

    return sorted(image_files)


# ============================================================
# АНАЛИЗ ОДНОГО ДАТАСЕТА
# ============================================================

def analyze_dataset(dataset_dir):

    dataset_name = dataset_dir.name

    images_dir = dataset_dir / "images"
    labels_dir = dataset_dir / "labels"


    if (
        not images_dir.exists()
        or not labels_dir.exists()
    ):

        return []


    # --------------------------------------------------------
    # Читаем исходные классы
    # --------------------------------------------------------

    yaml_classes = read_yaml_classes(
        dataset_dir
    )


    # --------------------------------------------------------
    # Исходный class_id ->
    # итоговый класс
    # --------------------------------------------------------

    id_mapping = {}


    for class_id, original_class in yaml_classes.items():

        key = (
            dataset_name,
            original_class
        )


        if key not in CLASS_MAPPING:

            continue


        id_mapping[class_id] = {

            "Исходный класс":
                original_class,

            "Итоговый класс":
                CLASS_MAPPING[key],

        }


    if not id_mapping:

        return []


    # --------------------------------------------------------
    # Счётчики
    # --------------------------------------------------------

    object_counts = defaultdict(int)

    image_sets = defaultdict(set)


    # --------------------------------------------------------
    # Изображения
    # --------------------------------------------------------

    image_files = get_image_files(
        images_dir
    )


    # --------------------------------------------------------
    # Читаем labels
    # --------------------------------------------------------

    for image_path in image_files:

        label_path = (
            labels_dir /
            f"{image_path.stem}.txt"
        )


        if not label_path.exists():

            continue


        try:

            with open(
                label_path,
                "r",
                encoding="utf-8"
            ) as f:

                lines = f.readlines()

        except Exception:

            continue


        for line in lines:

            parts = line.strip().split()


            if not parts:

                continue


            try:

                class_id = int(parts[0])

            except ValueError:

                continue


            # ------------------------------------------------
            # Класс не выбран
            # ------------------------------------------------

            if class_id not in id_mapping:

                continue


            original_class = (
                id_mapping[class_id]
                ["Исходный класс"]
            )


            final_class = (
                id_mapping[class_id]
                ["Итоговый класс"]
            )


            key = (
                dataset_name,
                original_class,
                final_class
            )


            # ------------------------------------------------
            # Объект
            # ------------------------------------------------

            object_counts[key] += 1


            # ------------------------------------------------
            # Изображение
            # ------------------------------------------------

            image_sets[key].add(
                str(image_path)
            )


    # --------------------------------------------------------
    # Формируем результат
    # --------------------------------------------------------

    rows = []


    for key in object_counts:

        (
            dataset_name,
            original_class,
            final_class
        ) = key


        rows.append({

            "Датасет":
                dataset_name,

            "Исходный класс":
                original_class,

            "Итоговый класс":
                final_class,

            "Количество объектов":
                object_counts[key],

            "Количество изображений":
                len(image_sets[key]),

        })


    return rows


# ============================================================
# АНАЛИЗ ВСЕХ ДАТАСЕТОВ
# ============================================================

print()
print("=" * 100)
print("НАЧАЛО АНАЛИЗА")
print("=" * 100)


all_rows = []


dataset_dirs = sorted([
    p
    for p in DATASETS_ROOT.iterdir()
    if p.is_dir()
])


for dataset_dir in dataset_dirs:

    print(
        f"Анализ: {dataset_dir.name}"
    )


    try:

        rows = analyze_dataset(
            dataset_dir
        )

        all_rows.extend(rows)


    except Exception as e:

        print(
            f"  ОШИБКА: {e}"
        )


# ============================================================
# ПРОВЕРКА
# ============================================================

selected_stats = pd.DataFrame(
    all_rows
)


if selected_stats.empty:

    raise RuntimeError(
        "Не найдено ни одного выбранного класса."
    )


# ============================================================
# СОРТИРОВКА
# ============================================================

selected_stats = selected_stats.sort_values(
    [
        "Итоговый класс",
        "Датасет",
        "Исходный класс"
    ]
).reset_index(
    drop=True
)


# ============================================================
# СОХРАНЕНИЕ ДЕТАЛЬНОЙ ТАБЛИЦЫ
# ============================================================

selected_stats.to_csv(

    OUTPUT_ROOT /
    "selected_classes_by_dataset.csv",

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# ВЫВОД №1
# ПОДРОБНАЯ СТАТИСТИКА
# ============================================================

print()
print("=" * 100)
print("РАСПРЕДЕЛЕНИЕ ВЫБРАННЫХ КЛАССОВ ПО ИСХОДНЫМ ДАТАСЕТАМ")
print("=" * 100)


display(
    selected_stats
)


# ============================================================
# УНИКАЛЬНЫЕ ИЗОБРАЖЕНИЯ ПО ИТОГОВЫМ КЛАССАМ
# ============================================================
#
# В отличие от простой суммы строк здесь одно изображение
# учитывается только один раз внутри итогового класса.
#
# ============================================================

final_class_images = defaultdict(set)


for dataset_dir in dataset_dirs:

    dataset_name = dataset_dir.name

    images_dir = dataset_dir / "images"
    labels_dir = dataset_dir / "labels"


    if (
        not images_dir.exists()
        or not labels_dir.exists()
    ):

        continue


    try:

        yaml_classes = read_yaml_classes(
            dataset_dir
        )

    except Exception:

        continue


    # --------------------------------------------------------
    # ID -> итоговый класс
    # --------------------------------------------------------

    id_mapping = {}


    for class_id, original_class in yaml_classes.items():

        key = (
            dataset_name,
            original_class
        )


        if key in CLASS_MAPPING:

            id_mapping[class_id] = (
                CLASS_MAPPING[key]
            )


    if not id_mapping:

        continue


    # --------------------------------------------------------
    # Изображения
    # --------------------------------------------------------

    image_files = get_image_files(
        images_dir
    )


    for image_path in image_files:

        label_path = (
            labels_dir /
            f"{image_path.stem}.txt"
        )


        if not label_path.exists():

            continue


        try:

            with open(
                label_path,
                "r",
                encoding="utf-8"
            ) as f:

                lines = f.readlines()

        except Exception:

            continue


        classes_in_image = set()


        for line in lines:

            parts = line.strip().split()


            if not parts:

                continue


            try:

                class_id = int(parts[0])

            except ValueError:

                continue


            if class_id not in id_mapping:

                continue


            classes_in_image.add(
                id_mapping[class_id]
            )


        # ----------------------------------------------------
        # Добавляем изображение только один раз
        # ----------------------------------------------------

        for final_class in classes_in_image:

            final_class_images[
                final_class
            ].add(
                (
                    dataset_name,
                    str(image_path)
                )
            )


# ============================================================
# СВОДКА ПО ИТОГОВЫМ КЛАССАМ
# ============================================================

summary_rows = []


for final_class, group in selected_stats.groupby(
    "Итоговый класс"
):


    summary_rows.append({

        "Итоговый класс":
            final_class,

        "Количество объектов":
            group[
                "Количество объектов"
            ].sum(),

        "Количество изображений":
            len(
                final_class_images.get(
                    final_class,
                    set()
                )
            ),

    })


summary = pd.DataFrame(
    summary_rows
)


summary = summary.sort_values(
    "Количество объектов",
    ascending=False
).reset_index(
    drop=True
)


# ============================================================
# ДОЛЯ ОБЪЕКТОВ
# ============================================================

total_objects = (
    summary[
        "Количество объектов"
    ].sum()
)


summary[
    "Доля объектов, %"
] = (

    summary[
        "Количество объектов"
    ]

    /

    total_objects

    * 100

).round(2)


# ============================================================
# ВЫВОД №2
# ОБЩАЯ СТАТИСТИКА
# ============================================================

print()
print("=" * 100)
print("СВОДНОЕ РАСПРЕДЕЛЕНИЕ ВЫБРАННЫХ КЛАССОВ")
print("=" * 100)


display(
    summary
)


summary.to_csv(

    OUTPUT_ROOT /
    "selected_classes_summary.csv",

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# КРАНЫ
# ============================================================
#
# ВАЖНО:
#
# Здесь балансировка производится ИМЕННО ПО КОЛИЧЕСТВУ
# ИЗОБРАЖЕНИЙ.
#
# Объекты используются только как дополнительная статистика.
#
# ============================================================


print()
print("=" * 100)
print("АНАЛИЗ ТРЁХ ТИПОВ КРАНОВ")
print("=" * 100)


crane_images = defaultdict(set)
crane_objects = defaultdict(int)


# ============================================================
# СБОР СТАТИСТИКИ КРАНОВ
# ============================================================

for dataset_dir in dataset_dirs:

    dataset_name = dataset_dir.name

    images_dir = dataset_dir / "images"
    labels_dir = dataset_dir / "labels"


    if (
        not images_dir.exists()
        or not labels_dir.exists()
    ):

        continue


    try:

        yaml_classes = read_yaml_classes(
            dataset_dir
        )

    except Exception:

        continue


    # --------------------------------------------------------
    # ID -> тип крана
    # --------------------------------------------------------

    crane_id_mapping = {}


    for class_id, original_class in yaml_classes.items():

        key = (
            dataset_name,
            original_class
        )


        if key not in CLASS_MAPPING:

            continue


        final_class = CLASS_MAPPING[key]


        if final_class in CRANE_CLASSES:

            crane_id_mapping[class_id] = (
                final_class
            )


    if not crane_id_mapping:

        continue


    # --------------------------------------------------------
    # Все изображения
    # --------------------------------------------------------

    image_files = get_image_files(
        images_dir
    )


    for image_path in image_files:

        label_path = (
            labels_dir /
            f"{image_path.stem}.txt"
        )


        if not label_path.exists():

            continue


        try:

            with open(
                label_path,
                "r",
                encoding="utf-8"
            ) as f:

                lines = f.readlines()

        except Exception:

            continue


        # ----------------------------------------------------
        # Типы кранов на конкретном изображении
        # ----------------------------------------------------

        crane_classes_in_image = set()


        for line in lines:

            parts = line.strip().split()


            if not parts:

                continue


            try:

                class_id = int(parts[0])

            except ValueError:

                continue


            if class_id not in crane_id_mapping:

                continue


            crane_class = (
                crane_id_mapping[class_id]
            )


            crane_classes_in_image.add(
                crane_class
            )


            # --------------------------------------------
            # Считаем объект
            # --------------------------------------------

            crane_objects[
                crane_class
            ] += 1


        # ----------------------------------------------------
        # Добавляем изображение
        #
        # Если на изображении несколько объектов одного
        # типа — оно всё равно считается ОДНИМ изображением.
        #
        # ----------------------------------------------------

        for crane_class in crane_classes_in_image:

            crane_images[
                crane_class
            ].add(
                (
                    dataset_name,
                    str(image_path)
                )
            )


# ============================================================
# ТАБЛИЦА №3
# КРАНЫ ДО БАЛАНСИРОВКИ
# ============================================================

crane_before_rows = []


for crane_class in CRANE_CLASSES:

    crane_before_rows.append({

        "Тип крана":
            crane_class,

        "Количество объектов":
            crane_objects.get(
                crane_class,
                0
            ),

        "Количество изображений":
            len(
                crane_images.get(
                    crane_class,
                    set()
                )
            ),

    })


crane_before = pd.DataFrame(
    crane_before_rows
)


print()
print("=" * 100)
print("КРАНЫ — ДО ОБЪЕДИНЕНИЯ И БАЛАНСИРОВКИ")
print("=" * 100)


display(
    crane_before
)


crane_before.to_csv(

    OUTPUT_ROOT /
    "crane_classes_before_balance.csv",

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# ПРОВЕРКА НАЛИЧИЯ ВСЕХ ТРЁХ ТИПОВ
# ============================================================

missing_crane_classes = []

for crane_class in CRANE_CLASSES:

    if (
        len(
            crane_images.get(
                crane_class,
                set()
            )
        )
        == 0
    ):

        missing_crane_classes.append(
            crane_class
        )


if missing_crane_classes:

    raise RuntimeError(
        "Для следующих типов кранов "
        "не найдено ни одного изображения:\n"
        +
        "\n".join(
            missing_crane_classes
        )
    )


# ============================================================
# ЦЕЛЕВОЕ КОЛИЧЕСТВО ИЗОБРАЖЕНИЙ
# ============================================================
#
# БЕРЁМ МИНИМАЛЬНОЕ КОЛИЧЕСТВО.
#
# Например:
#
# Подъёмный кран      3781
# Стационарный кран   3135
# Автокран            1791
#
# target_images = 1791
#
# ============================================================

target_images = int(
    crane_before[
        "Количество изображений"
    ].min()
)


# ============================================================
# ТАБЛИЦА №4
# КОНКРЕТНЫЙ ОБЪЁМ ДЛЯ СЛЕДУЮЩЕГО СКРИПТА
# ============================================================

crane_balance_rows = []


for _, row in crane_before.iterrows():

    crane_class = row[
        "Тип крана"
    ]


    objects_before = int(
        row[
            "Количество объектов"
        ]
    )


    images_before = int(
        row[
            "Количество изображений"
        ]
    )


    images_to_take = target_images


    images_to_remove = (
        images_before -
        images_to_take
    )


    crane_balance_rows.append({

        "Итоговый класс":
            CRANE_FINAL_CLASS,

        "Исходный тип крана":
            crane_class,

        "Объектов до балансировки":
            objects_before,

        "Изображений до балансировки":
            images_before,

        "Изображений взять":
            images_to_take,

        "Изображений исключить":
            images_to_remove,

    })


crane_balance = pd.DataFrame(
    crane_balance_rows
)


# ============================================================
# ВЫВОД №4
# ============================================================

print()
print("=" * 100)
print("КРАНЫ — ОБЪЁМ ДЛЯ ИТОГОВОГО DATASET")
print("=" * 100)


print()
print(
    f"Минимальное количество изображений: "
    f"{target_images}"
)


print()
print(
    "Следующий скрипт должен взять РОВНО "
    f"{target_images} изображений каждого типа крана."
)


display(
    crane_balance
)


crane_balance.to_csv(

    OUTPUT_ROOT /
    "crane_balance.csv",

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# СОХРАНЯЕМ КОНФИГУРАЦИЮ ДЛЯ СЛЕДУЮЩЕГО СКРИПТА
# ============================================================

crane_config = {

    "final_class":
        CRANE_FINAL_CLASS,

    "source_classes":
        CRANE_CLASSES,

    "balance_by":
        "images",

    "target_images_per_class":
        target_images,

}


with open(

    OUTPUT_ROOT /
    "crane_balance_config.yaml",

    "w",

    encoding="utf-8"

) as f:

    yaml.dump(

        crane_config,

        f,

        allow_unicode=True,

        sort_keys=False
    )


# ============================================================
# ИТОГОВАЯ ИНФОРМАЦИЯ
# ============================================================

print()
print("=" * 100)
print("ИТОГОВАЯ ИНФОРМАЦИЯ")
print("=" * 100)


print()


print(
    f"Выбрано итоговых классов: "
    f"{len(summary)}"
)


print(
    f"Всего объектов выбранных классов: "
    f"{int(summary['Количество объектов'].sum()):,}"
    .replace(",", " ")
)


all_selected_images = set()


for image_set in final_class_images.values():

    all_selected_images.update(
        image_set
    )


print(
    f"Всего уникальных изображений "
    f"с выбранными классами: "
    f"{len(all_selected_images):,}"
    .replace(",", " ")
)


print()


print(
    f"Краны будут объединены "
    f"в итоговый класс: "
    f"«{CRANE_FINAL_CLASS}»"
)


print(
    f"Балансировка кранов выполняется "
    f"по количеству изображений."
)


print(
    f"Целевое количество изображений "
    f"каждого типа крана: "
    f"{target_images}"
)


print(
    f"Всего изображений для трёх типов "
    f"кранов: "
    f"{target_images * 3}"
)


print()


print(
    "Файлы сохранены в:"
)

print(
    OUTPUT_ROOT
)


print()
print("=" * 100)
print("ГОТОВО")
print("=" * 100)


НАЧАЛО АНАЛИЗА
Анализ: -test- 2000-3000.v1i.yolo26
Анализ: 20230701_HeavyEquipment.v1i.yolo26
Анализ: constraction_yolo
Анализ: dataset_1
Анализ: dataset_2
Анализ: dataset_4
Анализ: FYP_MOCSdataset.v3i.yolo26_bbox
Анализ: Heavy Equipment_yolo
Анализ: large-machinery-v3.v1i.yolo26
Анализ: yolo_portable crane.v1i.yolo26

РАСПРЕДЕЛЕНИЕ ВЫБРАННЫХ КЛАССОВ ПО ИСХОДНЫМ ДАТАСЕТАМ


,Датасет,Исходный класс,Итоговый класс,Количество объектов,Количество изображений
0,-test- 2000-3000.v1i.yolo26,Pump truck,Автобетононасос,65,61
1,20230701_HeavyEquipment.v1i.yolo26,PumpTruck,Автобетононасос,364,327
2,FYP_MOCSdataset.v3i.yolo26_bbox,Pump truck,Автобетононасос,613,551
3,-test- 2000-3000.v1i.yolo26,Concrete mixer,Автобетоносмеситель,85,65
4,20230701_HeavyEquipment.v1i.yolo26,ConcreteMixer,Автобетоносмеситель,278,237
5,FYP_MOCSdataset.v3i.yolo26_bbox,Concrete mixer,Автобетоносмеситель,688,474
6,constraction_yolo,бетономешалка,Автобетоносмеситель,1118,976
7,dataset_1,Mixer,Автобетоносмеситель,1224,1180
8,dataset_1,Autocran,Автокран,1791,1791
9,-test- 2000-3000.v1i.yolo26,Bulldozer,Бульдозер,46,44



СВОДНОЕ РАСПРЕДЕЛЕНИЕ ВЫБРАННЫХ КЛАССОВ


,Итоговый класс,Количество объектов,Количество изображений,"Доля объектов, %"
0,Экскаватор,19923,15848,27.56
1,Самосвал,17500,12290,24.21
2,Грузовой автомобиль,6188,5253,8.56
3,Стационарный кран,5906,3135,8.17
4,Подъёмный кран,5877,3781,8.13
5,Фронтальный погрузчик,4857,3828,6.72
6,Автобетоносмеситель,3393,2932,4.69
7,Бульдозер,2766,2685,3.83
8,Прицеп,1830,1433,2.53
9,Автокран,1791,1791,2.48



АНАЛИЗ ТРЁХ ТИПОВ КРАНОВ

КРАНЫ — ДО ОБЪЕДИНЕНИЯ И БАЛАНСИРОВКИ


,Тип крана,Количество объектов,Количество изображений
0,Подъёмный кран,5877,3781
1,Стационарный кран,5906,3135
2,Автокран,1791,1791



КРАНЫ — ОБЪЁМ ДЛЯ ИТОГОВОГО DATASET

Минимальное количество изображений: 1791

Следующий скрипт должен взять РОВНО 1791 изображений каждого типа крана.


,Итоговый класс,Исходный тип крана,Объектов до балансировки,Изображений до балансировки,Изображений взять,Изображений исключить
0,Кран,Подъёмный кран,5877,3781,1791,1990
1,Кран,Стационарный кран,5906,3135,1791,1344
2,Кран,Автокран,1791,1791,1791,0



ИТОГОВАЯ ИНФОРМАЦИЯ

Выбрано итоговых классов: 12
Всего объектов выбранных классов: 72 281
Всего уникальных изображений с выбранными классами: 36 217

Краны будут объединены в итоговый класс: «Кран»
Балансировка кранов выполняется по количеству изображений.
Целевое количество изображений каждого типа крана: 1791
Всего изображений для трёх типов кранов: 5373

Файлы сохранены в:
C:\Users\Ева\Desktop\Хакатон\dataset

ГОТОВО


# Создание датасета

In [6]:
from pathlib import Path
from collections import defaultdict, Counter
import random
import shutil
import yaml
import pandas as pd

# ============================================================
# 1. ПУТИ
# ============================================================

DATASETS_ROOT = Path(
    r"C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets"
)

OUTPUT_ROOT = Path(
    r"C:\Users\Ева\Desktop\Хакатон\dataset"
)

# Если папка уже существует — удалить и создать заново
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

(OUTPUT_ROOT / "images" / "train").mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "images" / "val").mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "images" / "test").mkdir(parents=True, exist_ok=True)

(OUTPUT_ROOT / "labels" / "train").mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "labels" / "val").mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "labels" / "test").mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. НАСТРОЙКИ
# ============================================================

SEED = 42
random.seed(SEED)

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

# Максимум изображений для обычных классов
MAX_IMAGES_PER_CLASS = 3000

# Для кранов — строго по 1791 изображения
CRANE_IMAGES_PER_TYPE = 1791

CRANE_SOURCE_CLASSES = {
    "Подъёмный кран",
    "Стационарный кран",
    "Автокран",
}

CRANE_FINAL_CLASS = "Кран"


# ============================================================
# 3. ИТОГОВЫЕ КЛАССЫ
# ============================================================

FINAL_CLASSES = [
    "Экскаватор",
    "Самосвал",
    "Грузовой автомобиль",
    "Кран",
    "Фронтальный погрузчик",
    "Автобетоносмеситель",
    "Бульдозер",
    "Прицеп",
    "Каток",
    "Автобетононасос",
]

FINAL_CLASS_TO_ID = {
    name: idx
    for idx, name in enumerate(FINAL_CLASSES)
}

print("Итоговые классы:")
for class_name, class_id in FINAL_CLASS_TO_ID.items():
    print(f"{class_id}: {class_name}")


# ============================================================
# 4. МАППИНГ ИСХОДНЫХ КЛАССОВ
# ============================================================

CLASS_MAPPING = {

    # --------------------------------------------------------
    # -test- 2000-3000.v1i.yolo26
    # --------------------------------------------------------

    ("-test- 2000-3000.v1i.yolo26", "Bulldozer"):
        "Бульдозер",

    ("-test- 2000-3000.v1i.yolo26", "Concrete mixer"):
        "Автобетоносмеситель",

    ("-test- 2000-3000.v1i.yolo26", "Crane"):
        "Подъёмный кран",

    ("-test- 2000-3000.v1i.yolo26", "Excavator"):
        "Экскаватор",

    ("-test- 2000-3000.v1i.yolo26", "Pump truck"):
        "Автобетононасос",

    ("-test- 2000-3000.v1i.yolo26", "Roller"):
        "Каток",

    ("-test- 2000-3000.v1i.yolo26", "Static crane"):
        "Стационарный кран",

    ("-test- 2000-3000.v1i.yolo26", "Truck"):
        "Самосвал",


    # --------------------------------------------------------
    # 20230701_HeavyEquipment.v1i.yolo26
    # --------------------------------------------------------

    ("20230701_HeavyEquipment.v1i.yolo26", "Bulldozer"):
        "Бульдозер",

    ("20230701_HeavyEquipment.v1i.yolo26", "ConcreteMixer"):
        "Автобетоносмеситель",

    ("20230701_HeavyEquipment.v1i.yolo26", "Crane"):
        "Подъёмный кран",

    ("20230701_HeavyEquipment.v1i.yolo26", "Excavator"):
        "Экскаватор",

    ("20230701_HeavyEquipment.v1i.yolo26", "PumpTruck"):
        "Автобетононасос",

    ("20230701_HeavyEquipment.v1i.yolo26", "Roller"):
        "Каток",

    ("20230701_HeavyEquipment.v1i.yolo26", "StaticCrane"):
        "Стационарный кран",

    ("20230701_HeavyEquipment.v1i.yolo26", "Truck"):
        "Самосвал",


    # --------------------------------------------------------
    # constraction_yolo
    # --------------------------------------------------------

    ("constraction_yolo", "бетономешалка"):
        "Автобетоносмеситель",

    ("constraction_yolo", "грузовик"):
        "Самосвал",


    # --------------------------------------------------------
    # dataset_1
    # --------------------------------------------------------

    ("dataset_1", "Dump truck"):
        "Самосвал",

    ("dataset_1", "Excavator"):
        "Экскаватор",

    ("dataset_1", "Roller"):
        "Каток",

    ("dataset_1", "Gazelle"):
        "Грузовой автомобиль",

    ("dataset_1", "Bucket loader Big"):
        "Фронтальный погрузчик",

    ("dataset_1", "Mixer"):
        "Автобетоносмеситель",

    ("dataset_1", "Bulldozer"):
        "Бульдозер",

    ("dataset_1", "Truck"):
        "Грузовой автомобиль",

    ("dataset_1", "Trailer"):
        "Прицеп",

    ("dataset_1", "Bucket loader Standart"):
        "Фронтальный погрузчик",

    ("dataset_1", "Autocran"):
        "Автокран",


    # --------------------------------------------------------
    # dataset_2
    # --------------------------------------------------------

    ("dataset_2", "crane"):
        "Подъёмный кран",

    ("dataset_2", "excavator"):
        "Экскаватор",

    ("dataset_2", "truck"):
        "Самосвал",


    # --------------------------------------------------------
    # FYP_MOCSdataset.v3i.yolo26_bbox
    # --------------------------------------------------------

    ("FYP_MOCSdataset.v3i.yolo26_bbox", "Bulldozer"):
        "Бульдозер",

    ("FYP_MOCSdataset.v3i.yolo26_bbox", "Concrete mixer"):
        "Автобетоносмеситель",

    ("FYP_MOCSdataset.v3i.yolo26_bbox", "Crane"):
        "Подъёмный кран",

    ("FYP_MOCSdataset.v3i.yolo26_bbox", "Excavator"):
        "Экскаватор",

    ("FYP_MOCSdataset.v3i.yolo26_bbox", "Pump truck"):
        "Автобетононасос",

    ("FYP_MOCSdataset.v3i.yolo26_bbox", "Roller"):
        "Каток",

    ("FYP_MOCSdataset.v3i.yolo26_bbox", "Static crane"):
        "Стационарный кран",

    ("FYP_MOCSdataset.v3i.yolo26_bbox", "Truck"):
        "Самосвал",


    # --------------------------------------------------------
    # Heavy Equipment_yolo
    # --------------------------------------------------------

    ("Heavy Equipment_yolo", "bulldozer"):
        "Бульдозер",

    ("Heavy Equipment_yolo", "dump"):
        "Самосвал",

    ("Heavy Equipment_yolo", "excavator"):
        "Экскаватор",

    ("Heavy Equipment_yolo", "roller"):
        "Каток",


    # --------------------------------------------------------
    # large-machinery-v3.v1i.yolo26
    # --------------------------------------------------------

    ("large-machinery-v3.v1i.yolo26", "excavator"):
        "Экскаватор",

    ("large-machinery-v3.v1i.yolo26", "crane"):
        "Подъёмный кран",


    # --------------------------------------------------------
    # yolo_portable crane.v1i.yolo26
    # --------------------------------------------------------

    ("yolo_portable crane.v1i.yolo26", "Crane"):
        "Подъёмный кран",
}


# ============================================================
# 5. ПОИСК YAML
# ============================================================

def find_dataset_yaml(dataset_dir):
    candidates = [
        dataset_dir / "data.yaml",
        dataset_dir / "dataset.yaml",
    ]

    for path in candidates:
        if path.exists():
            return path

    # На случай другого названия
    yaml_files = list(dataset_dir.glob("*.yaml")) + list(dataset_dir.glob("*.yml"))

    if yaml_files:
        return yaml_files[0]

    return None


# ============================================================
# 6. ЧТЕНИЕ КЛАССОВ ИЗ YAML
# ============================================================

def read_yaml_classes(yaml_path):

    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    names = data.get("names", {})

    if isinstance(names, dict):
        names = {
            int(k): str(v)
            for k, v in names.items()
        }

    elif isinstance(names, list):
        names = {
            i: str(name)
            for i, name in enumerate(names)
        }

    else:
        raise ValueError(
            f"Не удалось прочитать names из {yaml_path}"
        )

    return names


# ============================================================
# 7. ПОИСК ИЗОБРАЖЕНИЙ
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


def get_image_files(images_dir):

    return [
        p
        for p in images_dir.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]


# ============================================================
# 8. СБОР ВСЕХ ДОСТУПНЫХ ОБЪЕКТОВ
# ============================================================

print("\n" + "=" * 80)
print("СБОР ИСХОДНЫХ ДАННЫХ")
print("=" * 80)

# Для каждого изображения:
#
# {
#     "image_path": ...,
#     "dataset": ...,
#     "objects": [
#         {
#             "source_class": ...,
#             "final_class": ...,
#             "line": ...
#         }
#     ]
# }

all_images = {}

# Статистика по классам
class_objects = Counter()
class_images = defaultdict(set)

# Статистика по исходным датасетам
dataset_statistics = defaultdict(
    lambda: {
        "images": 0,
        "objects": 0
    }
)


for dataset_dir in sorted(DATASETS_ROOT.iterdir()):

    if not dataset_dir.is_dir():
        continue

    dataset_name = dataset_dir.name

    yaml_path = find_dataset_yaml(dataset_dir)

    if yaml_path is None:
        print(f"[ПРОПУСК] Нет YAML: {dataset_name}")
        continue

    images_dir = dataset_dir / "images"
    labels_dir = dataset_dir / "labels"

    if not images_dir.exists() or not labels_dir.exists():
        print(f"[ПРОПУСК] Нет images/labels: {dataset_name}")
        continue

    try:
        class_names = read_yaml_classes(yaml_path)
    except Exception as e:
        print(f"[ОШИБКА YAML] {dataset_name}: {e}")
        continue

    image_files = get_image_files(images_dir)

    print(
        f"{dataset_name}: найдено {len(image_files)} изображений"
    )

    for image_path in image_files:

        label_path = labels_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            continue

        objects = []

        with open(label_path, "r", encoding="utf-8") as f:
            lines = [
                line.strip()
                for line in f
                if line.strip()
            ]

        for line in lines:

            parts = line.split()

            if len(parts) < 5:
                continue

            try:
                source_class_id = int(parts[0])
            except ValueError:
                continue

            if source_class_id not in class_names:
                continue

            source_class_name = class_names[source_class_id]

            key = (
                dataset_name,
                source_class_name
            )

            if key not in CLASS_MAPPING:
                continue

            final_class = CLASS_MAPPING[key]

            # Сохраняем объект
            objects.append({
                "source_class": source_class_name,
                "final_class": final_class,
                "line": line,
            })

            class_objects[final_class] += 1
            class_images[final_class].add(image_path)

            dataset_statistics[dataset_name]["objects"] += 1

        if objects:

            # Абсолютно уникальный ключ.
            # Это важно, потому что одинаковые имена файлов
            # могут существовать в разных датасетах.
            image_key = str(image_path.resolve())

            all_images[image_key] = {
                "image_path": image_path,
                "dataset": dataset_name,
                "label_path": label_path,
                "objects": objects,
            }

            dataset_statistics[dataset_name]["images"] += 1


print("\nВсего изображений с выбранными классами:",
      len(all_images))


# ============================================================
# 9. ИСХОДНАЯ СТАТИСТИКА
# ============================================================

print("\n" + "=" * 80)
print("ИСХОДНАЯ СТАТИСТИКА")
print("=" * 80)

for class_name in FINAL_CLASSES:

    if class_name == "Кран":
        continue

    print(
        f"{class_name:25s} "
        f"объектов: {class_objects[class_name]:6d} | "
        f"изображений: {len(class_images[class_name]):6d}"
    )

print("\nКраны:")

for crane_type in [
    "Автокран",
    "Подъёмный кран",
    "Стационарный кран",
]:

    print(
        f"{crane_type:25s} "
        f"объектов: {class_objects[crane_type]:6d} | "
        f"изображений: {len(class_images[crane_type]):6d}"
    )


# ============================================================
# 10. ОТБОР ИЗОБРАЖЕНИЙ
# ============================================================

print("\n" + "=" * 80)
print("ОТБОР ИЗОБРАЖЕНИЙ")
print("=" * 80)


# ------------------------------------------------------------
# 10.1 Обычные классы
# ------------------------------------------------------------

selected_images = set()

selection_statistics = []


for class_name in FINAL_CLASSES:

    if class_name == "Кран":
        continue

    available = list(class_images[class_name])

    random.shuffle(available)

    target = min(
        MAX_IMAGES_PER_CLASS,
        len(available)
    )

    selected = available[:target]

    selected_images.update(
        str(p.resolve())
        for p in selected
    )

    selection_statistics.append({
        "Класс": class_name,
        "Доступно изображений": len(available),
        "Целевое количество": target,
        "Выбрано": len(selected),
    })


# ------------------------------------------------------------
# 10.2 Краны
# ------------------------------------------------------------

# Для каждого типа крана независимо выбираем 1791 изображения.

selected_crane_images = {}

for crane_type in [
    "Автокран",
    "Подъёмный кран",
    "Стационарный кран",
]:

    available = list(class_images[crane_type])

    if len(available) < CRANE_IMAGES_PER_TYPE:

        raise ValueError(
            f"\nДля '{crane_type}' доступно только "
            f"{len(available)} изображений, "
            f"а требуется {CRANE_IMAGES_PER_TYPE}."
        )

    random.shuffle(available)

    selected = available[:CRANE_IMAGES_PER_TYPE]

    selected_crane_images[crane_type] = set(
        str(p.resolve())
        for p in selected
    )

    selected_images.update(
        selected_crane_images[crane_type]
    )

    selection_statistics.append({
        "Класс": crane_type,
        "Доступно изображений": len(available),
        "Целевое количество": CRANE_IMAGES_PER_TYPE,
        "Выбрано": len(selected),
    })


selection_df = pd.DataFrame(selection_statistics)

print(selection_df.to_string(index=False))

print(
    "\nУникальных изображений после объединения всех выбранных классов:",
    len(selected_images)
)


# ============================================================
# 11. ПРОВЕРКА ПЕРЕД SPLIT
# ============================================================

print("\n" + "=" * 80)
print("ПРОВЕРКА ВЫБРАННЫХ ИЗОБРАЖЕНИЙ")
print("=" * 80)


selected_class_images = defaultdict(set)
selected_class_objects = Counter()


for image_key in selected_images:

    item = all_images[image_key]

    for obj in item["objects"]:

        final_class = obj["final_class"]

        # Три типа кранов объединяем
        if final_class in CRANE_SOURCE_CLASSES:
            final_class = CRANE_FINAL_CLASS

        selected_class_images[final_class].add(
            image_key
        )

        selected_class_objects[final_class] += 1


for class_name in FINAL_CLASSES:

    print(
        f"{class_name:25s} "
        f"изображений: {len(selected_class_images[class_name]):6d} | "
        f"объектов: {selected_class_objects[class_name]:6d}"
    )


# ============================================================
# 12. SPLIT 80 / 10 / 10
# ============================================================

print("\n" + "=" * 80)
print("РАЗДЕЛЕНИЕ НА TRAIN / VAL / TEST")
print("=" * 80)


# Важно:
# split выполняется по УНИКАЛЬНЫМ изображениям,
# а не отдельно по классам.

selected_image_list = list(selected_images)

random.shuffle(selected_image_list)

n_total = len(selected_image_list)

n_test = round(n_total * TEST_RATIO)
n_val = round(n_total * VAL_RATIO)
n_train = n_total - n_val - n_test

train_images = selected_image_list[:n_train]

val_images = selected_image_list[
    n_train:n_train + n_val
]

test_images = selected_image_list[
    n_train + n_val:
]


splits = {
    "train": train_images,
    "val": val_images,
    "test": test_images,
}


print(f"Всего: {n_total}")
print(f"Train: {len(train_images)}")
print(f"Val:   {len(val_images)}")
print(f"Test:  {len(test_images)}")


# ============================================================
# 13. ПЕРЕНОС ФАЙЛОВ
# ============================================================

print("\n" + "=" * 80)
print("ФОРМИРОВАНИЕ ФИНАЛЬНОГО DATASET")
print("=" * 80)


def make_unique_filename(dataset_name, image_path):
    """
    Делает имя файла уникальным.
    Это защищает от ситуации, когда разные датасеты
    содержат image.jpg с одинаковым названием.
    """

    safe_dataset = (
        dataset_name
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
    )

    return f"{safe_dataset}__{image_path.name}"


actual_split_statistics = defaultdict(Counter)


for split_name, image_keys in splits.items():

    print(
        f"\nОбрабатывается {split_name}: "
        f"{len(image_keys)} изображений"
    )

    for image_key in image_keys:

        item = all_images[image_key]

        image_path = item["image_path"]
        dataset_name = item["dataset"]

        new_filename = make_unique_filename(
            dataset_name,
            image_path
        )

        output_image = (
            OUTPUT_ROOT
            / "images"
            / split_name
            / new_filename
        )

        output_label = (
            OUTPUT_ROOT
            / "labels"
            / split_name
            / f"{Path(new_filename).stem}.txt"
        )

        # ----------------------------------------------------
        # Копируем изображение
        # ----------------------------------------------------

        shutil.copy2(
            image_path,
            output_image
        )

        # ----------------------------------------------------
        # Формируем новую YOLO-разметку
        # ----------------------------------------------------

        new_label_lines = []

        for obj in item["objects"]:

            source_class = obj["source_class"]
            final_class = obj["final_class"]

            # Объединяем три типа крана
            if final_class in CRANE_SOURCE_CLASSES:
                final_class = CRANE_FINAL_CLASS

            # Если класс не входит в финальный набор — пропускаем
            if final_class not in FINAL_CLASS_TO_ID:
                continue

            new_class_id = FINAL_CLASS_TO_ID[final_class]

            parts = obj["line"].split()

            # Заменяем только ID класса
            parts[0] = str(new_class_id)

            new_label_lines.append(
                " ".join(parts)
            )

            actual_split_statistics[split_name][
                final_class
            ] += 1

        # ----------------------------------------------------
        # Сохраняем label
        # ----------------------------------------------------

        with open(
            output_label,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                "\n".join(new_label_lines)
            )


# ============================================================
# 14. ФАКТИЧЕСКАЯ СТАТИСТИКА TRAIN / VAL / TEST
# ============================================================

print("\n" + "=" * 80)
print("ФАКТИЧЕСКАЯ СТАТИСТИКА")
print("=" * 80)


final_statistics = []


for split_name in [
    "train",
    "val",
    "test",
]:

    image_dir = (
        OUTPUT_ROOT
        / "images"
        / split_name
    )

    label_dir = (
        OUTPUT_ROOT
        / "labels"
        / split_name
    )

    split_image_count = 0

    # Количество уникальных изображений,
    # содержащих каждый класс
    class_image_sets = defaultdict(set)

    # Количество объектов
    class_object_counts = Counter()

    for image_path in image_dir.iterdir():

        if not image_path.is_file():
            continue

        split_image_count += 1

        label_path = (
            label_dir
            / f"{image_path.stem}.txt"
        )

        if not label_path.exists():
            continue

        with open(
            label_path,
            "r",
            encoding="utf-8"
        ) as f:

            lines = [
                line.strip()
                for line in f
                if line.strip()
            ]

        classes_in_image = set()

        for line in lines:

            parts = line.split()

            if len(parts) < 5:
                continue

            class_id = int(parts[0])

            if class_id >= len(FINAL_CLASSES):
                continue

            class_name = FINAL_CLASSES[class_id]

            class_object_counts[class_name] += 1

            classes_in_image.add(class_name)

        for class_name in classes_in_image:
            class_image_sets[class_name].add(
                image_path.name
            )

    for class_name in FINAL_CLASSES:

        final_statistics.append({
            "Раздел": split_name,
            "Класс": class_name,
            "Изображений": len(
                class_image_sets[class_name]
            ),
            "Объектов": class_object_counts[class_name],
        })

    print(
        f"\n{split_name.upper()}: "
        f"{split_image_count} изображений"
    )

    for class_name in FINAL_CLASSES:

        print(
            f"  {class_name:25s} "
            f"images={len(class_image_sets[class_name]):5d}, "
            f"objects={class_object_counts[class_name]:6d}"
        )


final_statistics_df = pd.DataFrame(
    final_statistics
)


# ============================================================
# 15. СОХРАНЕНИЕ СТАТИСТИКИ
# ============================================================

selection_df.to_csv(
    OUTPUT_ROOT / "selection_statistics.csv",
    index=False,
    encoding="utf-8-sig"
)

final_statistics_df.to_csv(
    OUTPUT_ROOT / "final_statistics.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 16. СОЗДАНИЕ DATA.YAML
# ============================================================

data_yaml = {
    "path": str(OUTPUT_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",

    "nc": len(FINAL_CLASSES),

    "names": {
        i: name
        for i, name in enumerate(FINAL_CLASSES)
    }
}


with open(
    OUTPUT_ROOT / "data.yaml",
    "w",
    encoding="utf-8"
) as f:

    yaml.dump(
        data_yaml,
        f,
        allow_unicode=True,
        sort_keys=False
    )


# ============================================================
# 17. СОХРАНЕНИЕ ПАРАМЕТРОВ
# ============================================================

config = {
    "seed": SEED,

    "split": {
        "train": TRAIN_RATIO,
        "val": VAL_RATIO,
        "test": TEST_RATIO,
    },

    "max_images_per_class": MAX_IMAGES_PER_CLASS,

    "cranes": {
        "balance_by": "images",
        "images_per_type": CRANE_IMAGES_PER_TYPE,
        "source_classes": [
            "Автокран",
            "Подъёмный кран",
            "Стационарный кран",
        ],
        "final_class": "Кран",
    },

    "classes": FINAL_CLASSES,
}


with open(
    OUTPUT_ROOT / "dataset_config.yaml",
    "w",
    encoding="utf-8"
) as f:

    yaml.dump(
        config,
        f,
        allow_unicode=True,
        sort_keys=False
    )


# ============================================================
# 18. ИТОГ
# ============================================================

print("\n" + "=" * 80)
print("ГОТОВО")
print("=" * 80)

print(f"\nДатасет сохранён в:")
print(OUTPUT_ROOT)

print("\nСтруктура:")

print("""
dataset/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
│
├── labels/
│   ├── train/
│   ├── val/
│   └── test/
│
├── data.yaml
├── dataset_config.yaml
├── selection_statistics.csv
└── final_statistics.csv
""")

print(
    f"Train: {len(train_images)} изображений"
)

print(
    f"Val:   {len(val_images)} изображений"
)

print(
    f"Test:  {len(test_images)} изображений"
)

print(
    f"Всего: {len(selected_image_list)} изображений"
)

Итоговые классы:
0: Экскаватор
1: Самосвал
2: Грузовой автомобиль
3: Кран
4: Фронтальный погрузчик
5: Автобетоносмеситель
6: Бульдозер
7: Прицеп
8: Каток
9: Автобетононасос

СБОР ИСХОДНЫХ ДАННЫХ
-test- 2000-3000.v1i.yolo26: найдено 999 изображений
20230701_HeavyEquipment.v1i.yolo26: найдено 3962 изображений
constraction_yolo: найдено 3651 изображений
dataset_1: найдено 19982 изображений
dataset_2: найдено 318 изображений
dataset_4: найдено 319 изображений
FYP_MOCSdataset.v3i.yolo26_bbox: найдено 9238 изображений
Heavy Equipment_yolo: найдено 3876 изображений
large-machinery-v3.v1i.yolo26: найдено 2470 изображений
yolo_portable crane.v1i.yolo26: найдено 352 изображений

Всего изображений с выбранными классами: 36217

ИСХОДНАЯ СТАТИСТИКА
Экскаватор                объектов:  19923 | изображений:  15848
Самосвал                  объектов:  17500 | изображений:  12290
Грузовой автомобиль       объектов:   6188 | изображений:   5253
Фронтальный погрузчик     объектов:   4857 | изображений:  